# Step 3: Load and Filter 

In [93]:
import pandas as pd
df=pd.read_csv("WDI_CSV/WDICSV.csv")
df.shape

(396970, 70)

In [94]:
patterns = [
    r"^Gini index$",
    r"Income share held by lowest 20%",
    r"Income share held by highest 20%",
    r"Poverty headcount ratio at \$3\.00 a day",
    r"Unemployment, total \(% of total labor force\) \(modeled ILO estimate\)",
    r"GDP per capita \(current US\$\)"
]
pattern = "|".join(patterns)

df_filtered = df[df["Indicator Name"].str.contains(pattern, case=False, na=False, regex=True)]
df_filtered["Indicator Name"].unique()

<StringArray>
[                                       'GDP per capita (current US$)',
                                                          'Gini index',
                                    'Income share held by highest 20%',
                                     'Income share held by lowest 20%',
 'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)',
 'Unemployment, total (% of total labor force) (modeled ILO estimate)']
Length: 6, dtype: str

In [95]:
id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_cols = [col for col in df_filtered.columns if col not in id_vars]

df_long = df_filtered.melt(
    id_vars=id_vars, 
    value_vars=year_cols, 
    var_name="Year", 
    value_name="Value"
)
df_long.shape

(104940, 6)

# Step 4: Data Cleaning

In [96]:
df_long.head()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
0,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,1960,186.089515
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,1960,NaN
2,Africa Eastern and Southern,AFE,Income share held by highest 20%,SI.DST.05TH.20,1960,NaN
3,Africa Eastern and Southern,AFE,Income share held by lowest 20%,SI.DST.FRST.20,1960,NaN
4,Africa Eastern and Southern,AFE,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,1960,NaN


In [97]:
df_long.dtypes

Country Name          str
Country Code          str
Indicator Name        str
Indicator Code        str
Year                  str
Value             float64
dtype: object

In [98]:
df_long["Value"].isna().sum()

np.int64(71749)

In [99]:
df_clean = df_long.dropna(subset=["Value"])

In [100]:
df_clean["Year"] = df_clean["Year"].astype(int)

In [101]:
df_clean = df_clean.drop_duplicates()

In [102]:
df_clean.shape

(33191, 6)

In [103]:
df_clean_meta = pd.read_csv("WDI_CSV/WDICountry.csv")

In [104]:
country_meta = pd.read_csv("WDI_CSV/WDICountry.csv")
country_meta.head()

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,NaN,Latin America & Caribbean,High income,AW,...,NaN,Enhanced General Data Dissemination System (e-...,2020 (expected),NaN,NaN,Yes,NaN,NaN,2018.0,NaN
1,AFE,Africa Eastern and Southern,Africa Eastern and Southern,Africa Eastern and Southern,ZH,NaN,"26 countries, stretching from the Red Sea in t...",NaN,NaN,ZH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,The reporting period for national accounts dat...,Middle East & North Africa,Low income,AF,...,Consolidated central government,Enhanced General Data Dissemination System (e-...,1979,Multiple Indicator Cluster Survey 2022-2023,"Integrated household survey (IHS), 2016/17",NaN,2003.0,NaN,2018.0,NaN
3,AFW,Africa Western and Central,Africa Western and Central,Africa Western and Central,ZI,NaN,"22 countries, stretching from the westernmost ...",NaN,NaN,ZI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,The World Bank systematically assesses the app...,Sub-Saharan Africa,Lower middle income,AO,...,Budgetary central government,Enhanced General Data Dissemination System (e-...,2014,Inquérito de Indicadores Múltiplos e de Saúde ...,"Integrated household survey (IHS), 2008/09",NaN,2023.0,NaN,2018.0,NaN


In [105]:
country_meta.columns.tolist()

['Country Code',
 'Short Name',
 'Table Name',
 'Long Name',
 '2-alpha code',
 'Currency Unit',
 'Special Notes',
 'Region',
 'Income Group',
 'WB-2 code',
 'National accounts base year',
 'National accounts reference year',
 'SNA price valuation',
 'Lending category',
 'Other groups',
 'System of National Accounts',
 'Alternative conversion factor',
 'PPP survey year',
 'Balance of Payments Manual in use',
 'External debt Reporting status',
 'System of trade',
 'Government Accounting concept',
 'IMF data dissemination standard',
 'Latest population census',
 'Latest household survey',
 'Source of most recent Income and expenditure data',
 'Vital registration complete',
 'Latest agricultural census',
 'Latest industrial data',
 'Latest trade data',
 'Latest water withdrawal data']

In [106]:
df_clean = df_clean.merge(
    country_meta[["Country Code", "Region", "Income Group"]],
    on="Country Code",
    how="left"
)

In [107]:
df_clean["Type"] = df_clean["Region"].apply(lambda x: "Country" if pd.notna(x) else "Aggregate")

In [108]:
df_clean["Type"].value_counts()

Type
Country      28015
Aggregate     5176
Name: count, dtype: int64

In [109]:
df_clean.to_csv("Poverty_inequality_cleaned.csv", index=False)

In [110]:
df.loc[
    df["Indicator Name"].str.contains("GINI|Gini|Poverty headcount|Income share held by highest", case=False, na=False),
    "Indicator Name"
].unique()

<StringArray>
[                                                              'Gini index',
                                         'Income share held by highest 10%',
                                         'Income share held by highest 20%',
 'Multidimensional poverty headcount ratio (UNDP & OPHI) (% of population)',
  'Multidimensional poverty headcount ratio (World Bank) (% of population)',
      'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)',
      'Poverty headcount ratio at $4.20 a day (2021 PPP) (% of population)',
      'Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)',
      'Poverty headcount ratio at national poverty lines (% of population)',
       'Poverty headcount ratio at societal poverty line (% of population)']
Length: 10, dtype: str